# Model Comparison

## Setup / Настройка


In [ ]:
# База / core
import sys
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Sklearn / ML
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# Модуль проекта / project module
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "titanic_utils.py").exists())
sys.path.insert(0, str(ROOT / "src"))

import titanic_utils as tu

# Стиль / style
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 70)
plt.rcParams["figure.figsize"] = (8, 4)


In [57]:
df_model = pd.read_parquet(
    ROOT / "data" /  "titanic_fe.parquet"
)

X = df_model.drop(columns="survived")
y = df_model["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [58]:
# признаки для предобработки / features for preprocessing
numeric_features = (
    X_train
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(include=["object", "category", "string"])
    .columns
    .tolist()
)

In [59]:
# Предобработка / Preprocessing для деревьев 
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

In [60]:
# Предобработка / Preprocessing для логистической регрессии + стандартизация / standardization
logreg_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

In [61]:
models = {
    "LogisticRegression": Pipeline([
        ("preprocessor", logreg_preprocessor),
        ("model", LogisticRegression(max_iter=5000))
    ]),

    "RandomForest": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=1000,
            max_depth=8,
            min_samples_leaf=3,
            min_samples_split=10,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "ExtraTrees": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", ExtraTreesClassifier(
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "HistGradientBoosting": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", HistGradientBoostingClassifier(
            random_state=42
        ))
    ]),

    "CatBoost": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", CatBoostClassifier(
            verbose=0,
            random_state=42
        ))
    ]),

    "LightGBM": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", LGBMClassifier(
            random_state=42
        ))
    ]),

    "XGBoost": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", XGBClassifier(
            random_state=42
        ))
    ])
}

In [63]:
import time

results = []

for name, model in models.items():

    print("\n" + "=" * 50)
    print(name)

    start = time.time()

    try:

        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=5,
            scoring="accuracy",
            n_jobs=-1,
            error_score="raise"
        )

        elapsed = time.time() - start

        print(
            f"CV: {scores.mean():.4f} ± {scores.std():.4f}"
        )
        print(
            f"Time: {elapsed:.1f} sec"
        )

        results.append({
            "model": name,
            "cv_mean": scores.mean(),
            "cv_std": scores.std(),
            "time_sec": elapsed
        })

    except Exception as e:

        print("ERROR:")
        print(e)


results = (
    pd.DataFrame(results)
    .sort_values(
        by="cv_mean",
        ascending=False
    )
)

display(
    results.round(4)
)


LogisticRegression
CV: 0.8051 ± 0.0179
Time: 0.1 sec

RandomForest
CV: 0.8137 ± 0.0229
Time: 3.3 sec

ExtraTrees
CV: 0.7727 ± 0.0233
Time: 0.3 sec

HistGradientBoosting
ERROR:
Sparse data was passed for X, but dense data is required. Use '.toarray()' to convert to a dense numpy array.

CatBoost
CV: 0.7994 ± 0.0214
Time: 6.5 sec

LightGBM
[LightGBM] [Info] Number of positive: 320, number of negative: 518
[LightGBM] [Info] Number of positive: 320, number of negative: 517
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001152 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001058 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.

[LightGBM] [Info] Total Bins 559
[LightGBM] [Info] Total Bins 559
[LightG

/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/mchunikhin/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ERROR:
Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['0' '1']


,model,cv_mean,cv_std,time_sec
1,RandomForest,0.8137,0.0229,3.3365
0,LogisticRegression,0.8051,0.0179,0.0799
3,CatBoost,0.7994,0.0214,6.5443
4,LightGBM,0.7994,0.0305,3.5256
2,ExtraTrees,0.7727,0.0233,0.2968


In [64]:
y = y.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [65]:
hist_model = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                random_state=42
            )
        )
    ]
)

scores = cross_val_score(
    hist_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
    error_score="raise"
)

print(scores)
print()
print("Mean:", scores.mean())
print("Std :", scores.std())

TypeError: Sparse data was passed for X, but dense data is required. Use '.toarray()' to convert to a dense numpy array.